# Phase 2 — Exploratory Data Analysis & Feature Engineering

**Project:** Hospital Operations & Revenue Risk Intelligence Platform  
**Goal:** Understand hospital operations, financial performance, and data reliability before deploying AI models.

---

## Notebook Structure
1. Data Loading & Merge
2. Missing Value Analysis
3. Distribution Analysis
4. Outlier Detection
5. Feature Engineering
6. Target Variable Analysis
7. Export Modeling Dataset

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5),
                     'axes.titlesize': 13, 'axes.labelsize': 11})

# ── Load from SQLite (created in Phase 1) ────────────────────────────────────
conn = sqlite3.connect('../hospital.db')

patients_df = pd.read_sql('SELECT * FROM patients', conn, parse_dates=['registration_date'])
visits_df   = pd.read_sql('SELECT * FROM visits',   conn, parse_dates=['visit_date'])
billing_df  = pd.read_sql('SELECT * FROM billing',  conn, parse_dates=['billing_date'])
conn.close()

print(f'Patients: {patients_df.shape} | Visits: {visits_df.shape} | Billing: {billing_df.shape}')

## 1. Merge Dataset

In [ ]:
# ── Merge visits + billing + patients ────────────────────────────────────────
df = (visits_df
      .merge(billing_df,  on='visit_id',  how='left')
      .merge(patients_df, on='patient_id', how='left'))

print(f'Merged dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 2. Missing Value Analysis

> **Business context:** Missingness is not random — `payment_days` nulls correlate with claim status, which is a modeling signal in itself.

In [ ]:
# ── Overall null counts and percentages ──────────────────────────────────────
null_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct':   (df.isnull().sum() / len(df) * 100).round(2)
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=null_summary.reset_index(), y='index', x='missing_pct', ax=ax, palette='Reds_d')
ax.set_title('Missing Value % by Column')
ax.set_xlabel('Missing (%)'); ax.set_ylabel('Column')
plt.tight_layout(); plt.savefig('../data_outputs/eda_nulls.png', dpi=110); plt.show()
display(null_summary)

In [ ]:
# ── Null payment_days by claim_status ────────────────────────────────────────
# Hypothesis: nulls in payment_days are associated with Pending/Rejected claims
pd_null_by_status = df.groupby('claim_status')['payment_days'].apply(
    lambda x: x.isnull().sum()
).reset_index(name='null_payment_days')
pd_null_by_status['total'] = df.groupby('claim_status')['payment_days'].count().values
display(pd_null_by_status)

> **📊 Insight:** `payment_days` nulls are structurally tied to Pending/Rejected claims — not random missingness. This column should not be imputed naively; instead, a flag `payment_days_missing` will be engineered as a feature.

## 3. Distribution Analysis

In [ ]:
# ── Visit distribution by department ─────────────────────────────────────────
dept_counts = df['department'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dept_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted', len(dept_counts)))
axes[0].set_title('Visits by Department'); axes[0].set_xlabel(''); axes[0].tick_params(axis='x', rotation=45)

df['visit_type'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                      colors=sns.color_palette('pastel'))
axes[1].set_title('Visit Type Distribution'); axes[1].set_ylabel('')
plt.tight_layout(); plt.savefig('../data_outputs/eda_dept_visit.png', dpi=110); plt.show()

In [ ]:
# ── Distribution by insurance_provider and city ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['insurance_provider'].value_counts().plot(kind='bar', ax=axes[0],
    color=sns.color_palette('Set2', df['insurance_provider'].nunique()))
axes[0].set_title('Visits by Insurance Provider')
axes[0].tick_params(axis='x', rotation=45)

df['city'].value_counts().plot(kind='bar', ax=axes[1],
    color=sns.color_palette('Set3', df['city'].nunique()))
axes[1].set_title('Visits by City')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.savefig('../data_outputs/eda_insurer_city.png', dpi=110); plt.show()

In [ ]:
# ── Claim status distribution ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cs = df['claim_status'].value_counts()
cs.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c', '#f39c12'])
axes[0].set_title('Claim Status Distribution'); axes[0].tick_params(axis='x', rotation=0)

rs = df['risk_score'].value_counts()
rs.plot(kind='bar', ax=axes[1], color=['#3498db', '#e67e22', '#e74c3c'])
axes[1].set_title('Risk Score Distribution'); axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout(); plt.savefig('../data_outputs/eda_targets.png', dpi=110); plt.show()
print('Claim Status:\n', cs.to_dict())
print('\nRisk Score:\n', rs.to_dict())

> **📊 Insight:** Class imbalance is likely present in both target variables. Model B (claim outcome) will require SMOTE or class-weighting to prevent bias toward majority class.

In [ ]:
# ── Numeric distributions ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, color in zip(axes,
    ['billed_amount', 'length_of_stay_hours', 'payment_days'],
    ['steelblue', 'seagreen', 'tomato']):
    data = df[col].dropna()
    ax.hist(data, bins=50, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
    ax.axvline(data.median(), color='black', linestyle='--', label=f'Median: {data.median():.1f}')
    ax.legend(fontsize=8)

plt.tight_layout(); plt.savefig('../data_outputs/eda_numeric_dist.png', dpi=110); plt.show()

## 4. Outlier Detection

In [ ]:
# ── IQR-based outlier detection ───────────────────────────────────────────────
def iqr_outliers(series, label):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr    = q3 - q1
    lower  = q1 - 1.5 * iqr
    upper  = q3 + 1.5 * iqr
    n_out  = ((series < lower) | (series > upper)).sum()
    pct    = n_out / len(series) * 100
    print(f'  {label:30s}: {n_out:5,} outliers ({pct:.1f}%)  | bounds=[{lower:.1f}, {upper:.1f}]')
    return lower, upper

print('IQR Outlier Summary:')
bounds_billed  = iqr_outliers(df['billed_amount'].dropna(), 'billed_amount')
bounds_los     = iqr_outliers(df['length_of_stay_hours'].dropna(), 'length_of_stay_hours')
bounds_paydays = iqr_outliers(df['payment_days'].dropna(), 'payment_days')

In [ ]:
# ── Boxplots for outlier visualisation ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['billed_amount', 'length_of_stay_hours', 'payment_days']):
    df[col].dropna().plot(kind='box', ax=ax, vert=True,
                          boxprops=dict(color='steelblue'),
                          medianprops=dict(color='red', linewidth=2))
    ax.set_title(f'{col} Boxplot')

plt.tight_layout(); plt.savefig('../data_outputs/eda_boxplots.png', dpi=110); plt.show()

> **📊 Insight:** `billed_amount` has significant right-skew outliers — likely ICU or surgical cases with very high costs. These will be **retained** (they carry real clinical/financial signal) but **log-transformed** during feature engineering to reduce skew impact on linear models.

## 5. Feature Engineering

> This is the most critical phase — we derive business-relevant predictors from raw data.

In [ ]:
# ── 5.1 Time-based features ───────────────────────────────────────────────────
df['visit_month']      = df['visit_date'].dt.month
df['visit_dayofweek']  = df['visit_date'].dt.dayofweek   # 0=Mon, 6=Sun
df['visit_quarter']    = df['visit_date'].dt.quarter
df['is_weekend']       = (df['visit_dayofweek'] >= 5).astype(int)

# ── 5.2 Days since patient registration ──────────────────────────────────────
df['days_since_registration'] = (
    df['visit_date'] - df['registration_date']
).dt.days.clip(lower=0)

# ── 5.3 Per-patient visit frequency ──────────────────────────────────────────
visit_freq = df.groupby('patient_id')['visit_id'].count().rename('patient_visit_freq')
df = df.merge(visit_freq, on='patient_id', how='left')

# ── 5.4 Per-patient average LOS ───────────────────────────────────────────────
avg_los = (df.groupby('patient_id')['length_of_stay_hours']
             .mean()
             .rename('patient_avg_los'))
df = df.merge(avg_los, on='patient_id', how='left')

# ── 5.5 Insurance provider rejection rate ────────────────────────────────────
rejection = (df.groupby('insurance_provider')['claim_status']
               .apply(lambda x: (x == 'Rejected').sum() / len(x))
               .rename('insurer_rejection_rate'))
df = df.merge(rejection, on='insurance_provider', how='left')

# ── 5.6 Department average LOS ────────────────────────────────────────────────
dept_avg_los = (df.groupby('department')['length_of_stay_hours']
                  .mean()
                  .rename('dept_avg_los'))
df = df.merge(dept_avg_los, on='department', how='left')

# ── 5.7 LOS vs department average ratio ──────────────────────────────────────
df['los_vs_dept_avg'] = df['length_of_stay_hours'] / df['dept_avg_los'].replace(0, np.nan)

# ── 5.8 Bill amount per LOS hour ─────────────────────────────────────────────
df['bill_per_los_hour'] = df['billed_amount'] / df['length_of_stay_hours'].replace(0, np.nan)

# ── 5.9 Log-transformed billed amount ────────────────────────────────────────
df['log_billed_amount'] = np.log1p(df['billed_amount'])

# ── 5.10 Payment days missing flag ───────────────────────────────────────────
df['payment_days_missing'] = df['payment_days'].isnull().astype(int)

# ── 5.11 Approval ratio ───────────────────────────────────────────────────────
df['approval_ratio'] = (
    df['approved_amount'] / df['billed_amount'].replace(0, np.nan)
).fillna(0)

print(f'✅ Feature engineering complete. Dataset shape: {df.shape}')
print(f'New features added: visit_month, visit_dayofweek, visit_quarter, is_weekend,'
      f' days_since_registration, patient_visit_freq, patient_avg_los,'
      f' insurer_rejection_rate, dept_avg_los, los_vs_dept_avg,'
      f' bill_per_los_hour, log_billed_amount, payment_days_missing, approval_ratio')

## 6. Target Variable Analysis

In [ ]:
# ── Risk score vs key features ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# LOS by risk score
df.boxplot(column='length_of_stay_hours', by='risk_score', ax=axes[0])
axes[0].set_title('LOS by Risk Score'); axes[0].set_xlabel('Risk Score')

# Age by risk score
df.boxplot(column='age', by='risk_score', ax=axes[1])
axes[1].set_title('Age by Risk Score'); axes[1].set_xlabel('Risk Score')

# Billed amount by risk score
df.boxplot(column='log_billed_amount', by='risk_score', ax=axes[2])
axes[2].set_title('Log Billed Amount by Risk Score'); axes[2].set_xlabel('Risk Score')

plt.suptitle('')  # remove auto-title
plt.tight_layout(); plt.savefig('../data_outputs/eda_risk_features.png', dpi=110); plt.show()

In [ ]:
# ── Claim status vs key features ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Rejection rate by insurer
rej_by_dept = df.groupby('department')['claim_status'].apply(
    lambda x: (x == 'Rejected').sum() / len(x) * 100
).sort_values(ascending=False)
rej_by_dept.plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('Rejection Rate % by Department')
axes[0].tick_params(axis='x', rotation=45)

# Approval ratio distribution by claim status
df[df['claim_status'].isin(['Paid', 'Rejected', 'Pending'])].boxplot(
    column='approval_ratio', by='claim_status', ax=axes[1])
axes[1].set_title('Approval Ratio by Claim Status')
plt.suptitle('')
plt.tight_layout(); plt.savefig('../data_outputs/eda_claim_features.png', dpi=110); plt.show()

In [ ]:
# ── Correlation heatmap of numeric features ───────────────────────────────────
numeric_cols = ['age', 'length_of_stay_hours', 'billed_amount', 'approved_amount',
                'payment_days', 'log_billed_amount', 'patient_visit_freq',
                'patient_avg_los', 'insurer_rejection_rate', 'days_since_registration',
                'los_vs_dept_avg', 'approval_ratio']

corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlation Heatmap — Numeric Features')
plt.tight_layout(); plt.savefig('../data_outputs/eda_corr_heatmap.png', dpi=110); plt.show()

> **📊 Insight:** `billed_amount` and `approved_amount` are highly correlated for Paid claims, but diverge for Rejected. `approval_ratio` cleanly captures this signal. `patient_visit_freq` correlates with chronic conditions — a strong predictor for risk scoring.

## 7. Export Modeling Dataset

In [ ]:
import os, json
os.makedirs('../data_outputs', exist_ok=True)

# ── Select final modeling columns ─────────────────────────────────────────────
# Only include features that can be computed BEFORE the prediction target is known
# (no data leakage from approved_amount into claim_status prediction)

model_cols = [
    # Identifiers
    'visit_id', 'patient_id', 'visit_date',
    # Demographics
    'age', 'gender', 'city', 'chronic_flag',
    # Visit features
    'department', 'visit_type', 'length_of_stay_hours',
    'visit_month', 'visit_dayofweek', 'visit_quarter', 'is_weekend',
    # Doctor
    'doctor_id',
    # Billing features (pre-approval — safe for claim prediction)
    'billed_amount', 'log_billed_amount', 'billing_date',
    # Engineered features
    'days_since_registration', 'patient_visit_freq', 'patient_avg_los',
    'insurer_rejection_rate', 'dept_avg_los', 'los_vs_dept_avg',
    'bill_per_los_hour', 'payment_days_missing',
    # Insurance
    'insurance_provider',
    # Targets
    'risk_score', 'claim_status',
    # For evaluation only (not model input)
    'payment_days', 'approved_amount', 'approval_ratio'
]

model_df = df[model_cols].copy()
model_df.to_csv('../data_outputs/model_table.csv', index=False)
print(f'✅ model_table.csv saved: {model_df.shape}')

# ── Feature schema (for deployment validation) ────────────────────────────────
feature_schema = {
    'model_a_risk_features': [
        'age', 'gender', 'city', 'chronic_flag',
        'department', 'visit_type', 'length_of_stay_hours',
        'visit_month', 'visit_dayofweek', 'is_weekend',
        'patient_visit_freq', 'patient_avg_los',
        'dept_avg_los', 'los_vs_dept_avg',
        'days_since_registration'
    ],
    'model_b_claim_features': [
        'age', 'gender', 'city', 'chronic_flag',
        'department', 'visit_type', 'length_of_stay_hours',
        'billed_amount', 'log_billed_amount',
        'insurance_provider', 'insurer_rejection_rate',
        'visit_month', 'visit_dayofweek', 'is_weekend',
        'patient_visit_freq', 'bill_per_los_hour',
        'payment_days_missing'
    ],
    'target_model_a': 'risk_score',
    'target_model_b': 'claim_status',
    'categorical_cols': ['gender', 'city', 'department', 'visit_type', 'insurance_provider'],
    'numeric_cols': [
        'age', 'length_of_stay_hours', 'billed_amount', 'log_billed_amount',
        'patient_visit_freq', 'patient_avg_los', 'insurer_rejection_rate',
        'dept_avg_los', 'los_vs_dept_avg', 'bill_per_los_hour',
        'days_since_registration', 'visit_month', 'visit_dayofweek', 'is_weekend'
    ],
    'binary_cols': ['chronic_flag', 'payment_days_missing'],
    'version': '1.0',
    'created_at': pd.Timestamp.now().isoformat()
}

with open('../data_outputs/feature_schema.json', 'w') as f:
    json.dump(feature_schema, f, indent=2)
print('✅ feature_schema.json saved')

In [ ]:
# ── Final dataset summary ─────────────────────────────────────────────────────
print('MODEL TABLE SUMMARY')
print('='*50)
print(f'  Total records    : {len(model_df):,}')
print(f'  Total features   : {len(model_cols)}')
print(f'  Date range       : {model_df["visit_date"].min().date()} → {model_df["visit_date"].max().date()}')
print(f'  Risk score dist  : {model_df["risk_score"].value_counts().to_dict()}')
print(f'  Claim status dist: {model_df["claim_status"].value_counts().to_dict()}')
print('='*50)
print('\n✅ Phase 2 Complete. Proceed to Phase3_Modeling.ipynb')

---

## Phase 2 Summary

| Task | Outcome |
|---|---|
| Missing value analysis | `payment_days` nulls = structural (Pending/Rejected), not random |
| Distribution analysis | Imbalance confirmed in both target variables |
| Outlier detection | Right-skewed billed_amount → log-transformed |
| Feature engineering | 14 new features derived |
| Export | `model_table.csv` and `feature_schema.json` saved |

**Key modeling considerations flagged:**
- Class imbalance → apply class weights or resampling in Phase 3
- No data leakage: `approved_amount` excluded from Model B inputs
- Time-based split required (earliest 80% train, latest 20% test)